In [1]:
import os
import urllib.request
import functools
import pandas as pd
import numpy as np

In [2]:
DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

def download_file(url: str, dest_dir: str = DATA_DIR) -> str:
    filename = url.split("/")[-1]
    path = os.path.join(dest_dir, filename)
    if os.path.isfile(path):
        print(f"  [skip] {filename} already in {dest_dir}")
        return path
    print(f"  [down] {filename} ...", end=" ", flush=True)
    urllib.request.urlretrieve(url, path)
    print(f"done")
    return path

def load_xpt(source: str) -> pd.DataFrame:
    if source.startswith("http"):
        source = download_file(source)
    return pd.read_sas(source, format="xport")

def join_on_seqn(*dfs: pd.DataFrame) -> pd.DataFrame:
    """
    Only use this with data from the same survey cycle!
    """
    if not dfs:
        raise ValueError("At least one DataFrame is required.")
    return functools.reduce(
        lambda a, b: pd.merge(a, b, on="SEQN", how="outer"),
        dfs,
    )

def load_nhanes_cycle(files: dict[str, str]) -> pd.DataFrame:
    """
    `files` is a mapping of human-readable names to XPT URLs.
    """
    frames = []
    for name, url in files.items():
        try:
            df = load_xpt(url)
            frames.append(df)
            print(f"  [load] {name}: {len(df):,} rows, {df.shape[1]} cols")
        except Exception as exc:
            print(f"  [WARN] {name} could not be loaded — skipping. ({exc})")

    if not frames:
        raise RuntimeError("No NHANES files could be loaded.")

    merged = join_on_seqn(*frames)
    print(f"\n  Merged: {len(merged):,} rows, {merged.shape[1]} cols")
    return merged

# https://www.cdc.gov/nchs/linked-data/mortality-files/index.html
_MORTALITY_COLSPECS = [
    (0,  14),   # SEQN          - participant ID
    (14, 15),   # ELIGSTAT      - eligibility (1=elig, 2=under 18, 3=inelig)
    (15, 16),   # MORTSTAT      - mortality (0=assumed alive, 1=deceased)
    (16, 19),   # UCOD_LEADING  - underlying cause of death (ICD-10 group)
    (19, 20),   # DIABETES      - diabetes as contributing cause
    (20, 21),   # HYPERTEN      - hypertension as contributing cause
    (42, 45),   # PERMTH_INT    - person-months of follow-up (interview)
    (45, 48),   # PERMTH_EXM    - person-months of follow-up (exam)
]
_MORTALITY_NAMES = [
    "SEQN", "ELIGSTAT", "MORTSTAT", "UCOD_LEADING",
    "DIABETES", "HYPERTEN", "PERMTH_INT", "PERMTH_EXM",
]

def load_linked_mortality(source: str) -> pd.DataFrame:
    df = pd.read_fwf(
        source,
        colspecs=_MORTALITY_COLSPECS,
        names=_MORTALITY_NAMES,
        na_values=["", "."],
    )
    print(f"  [load] mortality: {len(df):,} rows")
    return df

In [3]:
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/"
CYCLE_FILES = {
    "DEMO":   BASE + "DEMO_J.xpt",    # demographics
    "BPXO":   BASE + "BPXO_J.xpt",    # blood pressure (oscillometric)
    "BPX":    BASE + "BPX_J.xpt",     # blood pressure (older format, fallback)
    "BMX":    BASE + "BMX_J.xpt",     # body measures
    "TCHOL":  BASE + "TCHOL_J.xpt",   # total cholesterol
    "HDL":    BASE + "HDL_J.xpt",     # HDL cholesterol
    "TRIGLY": BASE + "TRIGLY_J.xpt",  # triglycerides + LDL
    "GHB":    BASE + "GHB_J.xpt",     # glycohemoglobin (HbA1c)
    "GLU":    BASE + "GLU_J.xpt",     # fasting glucose
    "INS":    BASE + "INS_J.xpt",     # fasting insulin
    "HSCRP":  BASE + "HSCRP_J.xpt",   # high-sensitivity CRP
    "BIOPRO": BASE + "BIOPRO_J.xpt",  # biochemistry (creatinine, BUN, uric acid)
    "ALB_CR": BASE + "ALB_CR_J.xpt",  # albumin/creatinine ratio
    "CBC":    BASE + "CBC_J.xpt",     # complete blood count
    "DIQ":    BASE + "DIQ_J.xpt",     # diabetes questionnaire
    "BPQ":    BASE + "BPQ_J.xpt",     # blood pressure questionnaire
    "MCQ":    BASE + "MCQ_J.xpt",     # medical conditions (label source)
    "KIQ_U":  BASE + "KIQ_U_J.xpt",   # kidney conditions
    "SMQ":    BASE + "SMQ_J.xpt",     # smoking questionnaire
    "ALQ":    BASE + "ALQ_J.xpt",     # alcohol questionnaire
    "PAQ":    BASE + "PAQ_J.xpt",     # physical activity
    "SLQ":    BASE + "SLQ_J.xpt",     # sleep questionnaire
}

df = load_nhanes_cycle(CYCLE_FILES)
df

  [skip] DEMO_J.xpt already in ./data
  [load] DEMO: 9,254 rows, 46 cols
  [skip] BPXO_J.xpt already in ./data
  [load] BPXO: 7,132 rows, 13 cols
  [skip] BPX_J.xpt already in ./data
  [load] BPX: 8,704 rows, 21 cols
  [skip] BMX_J.xpt already in ./data
  [load] BMX: 8,704 rows, 21 cols
  [skip] TCHOL_J.xpt already in ./data
  [load] TCHOL: 7,435 rows, 3 cols
  [skip] HDL_J.xpt already in ./data
  [load] HDL: 7,435 rows, 3 cols
  [skip] TRIGLY_J.xpt already in ./data
  [load] TRIGLY: 3,036 rows, 10 cols
  [skip] GHB_J.xpt already in ./data
  [load] GHB: 6,401 rows, 2 cols
  [skip] GLU_J.xpt already in ./data
  [load] GLU: 3,036 rows, 4 cols
  [skip] INS_J.xpt already in ./data
  [load] INS: 3,036 rows, 5 cols
  [skip] HSCRP_J.xpt already in ./data
  [load] HSCRP: 8,366 rows, 3 cols
  [skip] BIOPRO_J.xpt already in ./data
  [load] BIOPRO: 6,401 rows, 41 cols
  [skip] ALB_CR_J.xpt already in ./data
  [load] ALB_CR: 7,936 rows, 8 cols
  [skip] CBC_J.xpt already in ./data
  [load] CBC: 8,3

,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,SLQ300,SLQ310,SLD012,SLQ320,SLQ330,SLD013,SLQ030,SLQ040,SLQ050,SLQ120
0,93703.0,10.0,2.0,2.0,2.0,NaN,5.0,6.0,2.0,27.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,93704.0,10.0,2.0,1.0,2.0,NaN,3.0,3.0,1.0,33.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,93705.0,10.0,2.0,2.0,66.0,NaN,4.0,4.0,2.0,NaN,...,b'23:00',b'07:00',8.0,b'23:00',b'07:00',8.0,2.000000e+00,5.397605e-79,2.0,5.397605e-79
3,93706.0,10.0,2.0,1.0,18.0,NaN,5.0,6.0,2.0,222.0,...,b'23:30',b'10:00',10.5,b'00:30',b'12:00',11.5,1.000000e+00,5.397605e-79,2.0,1.000000e+00
4,93707.0,10.0,2.0,1.0,13.0,NaN,5.0,7.0,2.0,158.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9249,102952.0,10.0,2.0,2.0,70.0,NaN,5.0,6.0,2.0,NaN,...,b'22:30',b'07:00',8.5,b'22:30',b'07:00',8.5,5.397605e-79,5.397605e-79,2.0,5.397605e-79
9250,102953.0,10.0,2.0,1.0,42.0,NaN,1.0,1.0,2.0,NaN,...,b'22:00',b'04:00',6.0,b'23:00',b'04:00',5.0,5.397605e-79,1.000000e+00,1.0,2.000000e+00
9251,102954.0,10.0,2.0,2.0,41.0,NaN,4.0,4.0,1.0,NaN,...,b'22:00',b'06:00',8.0,b'00:00',b'07:00',7.0,1.000000e+00,5.397605e-79,2.0,1.000000e+00
9252,102955.0,10.0,2.0,2.0,14.0,NaN,4.0,4.0,2.0,175.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
MIN_AGE      = 40  # CVD questions are only asked of adults 40+

# MCQ160E = ever told had heart attack, MCQ160F = ever told had stroke
# Value coding: 1=Yes, 2=No, 7=Refused, 9=Don't know
LABEL_COLS = ["MCQ160E", "MCQ160F"]

def build_label(df: pd.DataFrame):
    present = [c for c in LABEL_COLS if c in df.columns]
    if not present:
        raise ValueError(
            "MCQ file is required (contains the CVD label columns). "
            "Check that MCQ_J.xpt downloaded correctly."
        )
    y = (df[present] == 1).any(axis=1).astype(int)
    # Keep only rows where at least one label column was answered
    valid = df[present].notna().any(axis=1)
    return y, valid

def _col(df, name):
    """Return the column if present, else an all-NaN Series."""
    return df[name] if name in df.columns else pd.Series(np.nan, index=df.index)

def _yesno(series):
    """Recode NHANES yes/no: 1→1, 2→0, refused/DK→NaN."""
    return series.replace({2: 0, 7: np.nan, 9: np.nan})

def _mean_bp(df, prefixes):
    """Average up to three BP readings; return NaN if none present."""
    cols = [c for c in df.columns if any(c.startswith(p) for p in prefixes)]
    return df[cols].mean(axis=1) if cols else pd.Series(np.nan, index=df.index)

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    f = {}

    # Demographics
    f["age"]       = _col(df, "RIDAGEYR")
    f["female"]    = (_col(df, "RIAGENDR") == 2).astype(float)
    f["race"]      = _col(df, "RIDRETH3")
    f["education"] = _col(df, "DMDEDUC2").replace({7: np.nan, 9: np.nan})
    f["income"]    = _col(df, "INDFMPIR")

    # Blood pressure — try oscillometric readings first (BPXO), fall back to BPX
    if any(c.startswith("BPXOSY") for c in df.columns):
        f["sbp"] = _mean_bp(df, ["BPXOSY"])
        f["dbp"] = _mean_bp(df, ["BPXODI"])
        f["pulse"] = _col(df, "BPXOPLS")
    else:
        f["sbp"] = _mean_bp(df, ["BPXSY"])
        f["dbp"] = _mean_bp(df, ["BPXDI"])
        f["pulse"] = _col(df, "BPXPLS")

    # Body measures
    f["bmi"]   = _col(df, "BMXBMI")
    f["waist"] = _col(df, "BMXWAIST")

    # Lipids
    tc  = _col(df, "LBXTC")
    hdl = _col(df, "LBDHDD")
    f["total_chol"] = tc
    f["hdl"]        = hdl
    f["ldl"]        = _col(df, "LBDLDL")
    f["trig"]       = _col(df, "LBXTR")
    f["non_hdl"]    = tc - hdl       # derived atherogenic fraction

    # Metabolic / labs
    f["hba1c"]      = _col(df, "LBXGH")
    f["glucose"]    = _col(df, "LBXGLU")
    f["insulin"]    = _col(df, "LBXIN")
    f["crp"]        = _col(df, "LBXHSCRP")
    f["creatinine"] = _col(df, "LBXSCR")
    f["bun"]        = _col(df, "LBXSBU")
    f["uric_acid"]  = _col(df, "LBXSUA")
    f["uacr"]       = _col(df, "URDACT")
    f["wbc"]        = _col(df, "LBXWBCSI")

    # Diagnoses / risk history (not CVD itself)
    f["diabetes_dx"]  = _yesno(_col(df, "DIQ010"))
    f["htn_dx"]       = _yesno(_col(df, "BPQ020"))
    f["highchol_dx"]  = _yesno(_col(df, "BPQ080"))
    f["family_hx"]    = _yesno(_col(df, "MCQ300A"))
    f["kidney_dx"]    = _yesno(_col(df, "KIQ022"))

    # Smoking: 0=never, 1=former, 2=current
    smoked = _col(df, "SMQ020")
    now    = _col(df, "SMQ040")
    smoke  = pd.Series(np.nan, index=df.index)
    smoke[smoked == 2]                    = 0   # never smoked 100+ cigarettes
    smoke[(smoked == 1) & (now == 3)]     = 1   # former smoker
    smoke[(now == 1) | (now == 2)]        = 2   # current smoker
    f["smoking"] = smoke

    # Lifestyle
    f["alcohol_day"]   = _col(df, "ALQ130").replace({777: np.nan, 999: np.nan})
    f["sedentary_min"] = _col(df, "PAD680").replace({7777: np.nan, 9999: np.nan})
    f["sleep_hours"]   = _col(df, "SLD012")

    X = pd.DataFrame(f, index=df.index)
    # Drop features whose source file was entirely absent (all-NaN columns)
    X = X.dropna(axis=1, how="all")
    return X

def prepare(df: pd.DataFrame):
    y, valid = build_label(df)
    X = build_features(df)
    keep = valid & (X["age"] >= MIN_AGE)
    print(f"  After age ≥{MIN_AGE} + label filter: {keep.sum():,} rows")
    return X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)

In [7]:
store = pd.HDFStore('data/store.h5')
store['df'] = df
store['X'], store['y'] = prepare(df)
store.close()

  After age ≥40 + label filter: 3,882 rows


/tmp/ipykernel_3319293/1269630737.py:2: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block1_values] [items->Index(['BPAOARM', 'SMDUPCA', 'SMD100BR', 'SLQ300', 'SLQ310', 'SLQ320',
       'SLQ330'],
      dtype='str')]

  store['df'] = df
